In [0]:
# Read the raw product category translation CSV using Auto Loader
# with schema inference enabled

df_category_translation_bronze = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/Volumes/second_data_engineering_project/bronze/schemas/product_category_translation") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option("header", "true") \
    .load("/Volumes/second_data_engineering_project/landing/raw_files/")

In [0]:
# With Auto Loader streaming, we can inspect the inferred schema

df_category_translation_bronze.printSchema()

In [0]:
# Write the streaming Bronze DataFrame as a Delta table
# Using Trigger.AvailableNow for batch-like processing with Auto Loader benefits
# Checkpoint location enables incremental processing on subsequent runs

df_category_translation_bronze.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/second_data_engineering_project/bronze/checkpoints/product_category_translation") \
    .trigger(availableNow=True) \
    .toTable("second_data_engineering_project.bronze.product_category_name_translation")

In [0]:
%sql
-- Display sample data from the Bronze product_category_name_translation table

SELECT *
FROM second_data_engineering_project.bronze.product_category_name_translation
LIMIT 100;

In [0]:
%sql
-- View actual rescued data if any exists

SELECT *
FROM second_data_engineering_project.bronze.product_category_name_translation
WHERE _rescued_data IS NOT NULL
LIMIT 100;

In [0]:
%sql
-- Validate the Auto Loader output by counting rows in the Bronze product_category_name_translation table

SELECT COUNT(*) AS row_count
FROM second_data_engineering_project.bronze.product_category_name_translation;